# Exercise 03. Aggregations

In [1]:
import pandas as pd 
import sqlite3 

conn = sqlite3.connect('../data/checking-logs.sqlite')

## Checking test table 

In [2]:
pd.read_sql("PRAGMA table_info(test);", conn)

,cid,name,type,notnull,dflt_value,pk
0,0,uid,TEXT,0,None,0
1,1,labname,TEXT,0,None,0
2,2,first_commit_ts,TIMESTAMP,0,None,0
3,3,first_view_ts,TIMESTAMP,0,None,0


## Getting first 10 rows

In [3]:
pd.read_sql("SELECT * FROM test LIMIT 10;", conn)

,uid,labname,first_commit_ts,first_view_ts
0,user_1,laba04,2020-04-26 17:06:18.462708,2020-04-26 21:53:59.624136
1,user_1,laba04s,2020-04-26 17:12:11.843671,2020-04-26 21:53:59.624136
2,user_1,laba05,2020-05-02 19:15:18.540185,2020-04-26 21:53:59.624136
3,user_1,laba06,2020-05-17 16:26:35.268534,2020-04-26 21:53:59.624136
4,user_1,laba06s,2020-05-20 12:23:37.289724,2020-04-26 21:53:59.624136
5,user_1,project1,2020-05-14 20:56:08.898880,2020-04-26 21:53:59.624136
6,user_10,laba04,2020-04-25 08:24:52.696624,2020-04-18 12:19:50.182714
7,user_10,laba04s,2020-04-25 08:37:54.604222,2020-04-18 12:19:50.182714
8,user_10,laba05,2020-05-01 19:27:26.063245,2020-04-18 12:19:50.182714
9,user_10,laba06,2020-05-19 11:39:28.885637,2020-04-18 12:19:50.182714


## Finding the minimum value

In [4]:
df_min = pd.read_sql("""
SELECT t.uid,
MIN((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600.0) AS min_diff
FROM test t 
JOIN deadlines d
ON t.labname = d.labs 
WHERE t.labname !='project1'
GROUP BY t.uid 
ORDER BY min_diff DESC                    
""", conn)
df_min #given in hours

,uid,min_diff
0,user_18,-10.973611
1,user_17,-81.591667
2,user_21,-126.199722
3,user_10,-132.341944
4,user_19,-148.916111
5,user_25,-150.870000
6,user_28,-174.853056
7,user_1,-175.556667
8,user_3,-182.055278
9,user_14,-200.766389


## Finding the maximum value

In [5]:
df_max = pd.read_sql("""
SELECT t.uid,
MAX((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600.0) AS max_diff
FROM test t 
JOIN deadlines d
ON t.labname = d.labs 
WHERE t.labname !='project1'
GROUP BY t.uid 
ORDER BY max_diff ASC
""", conn)
df_max #given in hours

,uid,max_diff
0,user_14,-84.448611
1,user_3,-60.511667
2,user_30,-52.478056
3,user_10,-39.368056
4,user_17,-34.643056
5,user_21,-33.905278
6,user_19,-32.729444
7,user_28,-8.104167
8,user_1,-6.796667
9,user_18,-3.934167


## Average

In [6]:
df_avg = pd.read_sql("""
SELECT AVG((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600.0) AS avg_diff
FROM test t 
JOIN deadlines d 
ON t.labname = d.labs
WHERE t.labname !='project1'
""", conn)
df_avg #given in hours

,avg_diff
0,-89.687841


## Pageviews vs time difference

In [7]:
views_diff = pd.read_sql("""
SELECT t.uid,
AVG((strftime('%s', t.first_commit_ts) - d.deadlines) / 3600.0) AS avg_diff,
COUNT(p.uid) AS pageviews
FROM test t
JOIN deadlines d ON t.labname = d.labs
LEFT JOIN pageviews p ON t.uid = p.uid
WHERE t.labname != 'project1'
GROUP BY t.uid
ORDER BY avg_diff ASC
""", conn)
views_diff #given in hours



,uid,avg_diff,pageviews
0,user_14,-159.568796,429
1,user_30,-145.528681,12
2,user_3,-105.738222,1585
3,user_19,-99.440417,64
4,user_21,-96.111181,40
5,user_25,-93.474944,895
6,user_28,-86.793833,745
7,user_10,-75.242444,445
8,user_1,-65.119778,140
9,user_17,-62.207667,235


## Checking correlation

In [8]:
views_diff[['avg_diff', 'pageviews']].corr()

,avg_diff,pageviews
avg_diff,1.000000,-0.185042
pageviews,-0.185042,1.000000


In [9]:
# self learning for correlation
# visualization of correlation between average ours before deadline vs times of visits in newsfeed

# import matplotlib.pyplot as plt

# views_diff.plot.scatter(x='pageviews', y='avg_diff')
# plt.xlabel("Newsfeed visits")
# plt.ylabel("Average hours before deadline")
# plt.show()

## Closing the connection

In [10]:
conn.close()